# [Phase 2-2: OpenAI 임베딩 + Steam API 테스트]

In [1]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import requests
from openai import OpenAI
from dotenv import load_dotenv
import time

# 1. 환경 변수 로드 (.env에 저장한 키들 불러오기)
load_dotenv()
STEAM_KEY = os.getenv("STEAM_API_KEY")
OPENAI_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_KEY)

# 2. 테스트용: 스팀의 모든 게임 목록 중 상위 5개만 맛보기 (Web API)
def get_steam_app_list():
    try:
        url = f"http://api.steampowered.com/ISteamApps/GetAppList/v2/?key={STEAM_KEY}"
        response = requests.get(url)
        
        # 상태 코드가 200(성공)이 아니면 에러 출력
        if response.status_code != 200:
            print(f"❌ 스팀 목록 호출 실패! 상태 코드: {response.status_code}")
            return []
            
        return response.json()['applist']['apps'][:10] # 일단 10개만 테스트
    except Exception as e:
        print(f"❌ 목록 가져오기 중 예외 발생: {e}")
        return []







def get_game_detail(app_id):
    try:
        url = f"https://store.steampowered.com/api/appdetails?appids={app_id}&l=korean"
        response = requests.get(url)
        
        # JSON으로 변환하기 전에 데이터가 있는지 먼저 확인
        if not response.text.strip(): 
            return None
            
        res = response.json()
        if res and str(app_id) in res and res[str(app_id)]['success']:
            return res[str(app_id)]['data']
    except Exception as e:
        # JSONDecodeError 등을 여기서 잡아냅니다.
        print(f"⚠️ ID {app_id} 상세정보 파싱 실패 (넘어갑니다)")
        return None
    return None

# --- 실행 테스트 ---
apps = get_steam_app_list()
if not apps:
    print("😭 가져온 앱 목록이 없습니다. API KEY를 다시 확인해 보세요!")
else:
    for app in apps:
        detail = get_game_detail(app['appid'])
        if detail:
            print(f"✅ 수집 성공: {detail['name']}")
        else:
            print(f"⏩ 스킵됨: {app['name']} (상세 정보 없음)")
        
        # 💡 시니어의 팁: 스팀 서버를 배려해 0.5초씩 쉬어줍니다. (Rate Limit 방지)
        time.sleep(0.5)

❌ 스팀 목록 호출 실패! 상태 코드: 404
😭 가져온 앱 목록이 없습니다. API KEY를 다시 확인해 보세요!


In [4]:
import os
import requests
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
STEAM_KEY = os.getenv("STEAM_API_KEY")
OPENAI_KEY = os.getenv("OPENAI_API_KEY")

print("🔍 키 유효성 검사를 시작합니다...\n")

# 1. OpenAI 키 확인 (모델 목록 불러오기로 테스트)
try:
    client = OpenAI(api_key=OPENAI_KEY)
    # 단순히 모델 리스트를 요청해봅니다. 키가 틀리거나 돈이 없으면 여기서 에러가 납니다.
    client.models.list()
    print("✅ OpenAI API 키: 정상 (사용 가능)")
except Exception as e:
    print(f"❌ OpenAI API 키: 오류 발생! ({e})")
    print("   -> 결제 수단 등록 여부나 키 복사 상태를 확인하세요.")

print("-" * 30)

# 2. Steam API 키 확인 (간단한 공용 호출로 테스트)
try:
    # GetAppList는 키가 없어도 되지만, ?key= 를 붙여서 호출했을 때 200이 나오면 키가 유효한 것입니다.
    test_url = f"https://api.steampowered.com/ISteamApps/GetAppList/v2/?key={STEAM_KEY}"
    response = requests.get(test_url, timeout=5)
    
    if response.status_code == 200:
        print("✅ Steam API 키: 정상 (사용 가능)")
    elif response.status_code == 403:
        print("❌ Steam API 키: 권한 거부 (403 Forbidden)")
        print("   -> 키가 잘못되었거나 도메인 등록 문제일 수 있습니다.")
    else:
        print(f"⚠️ Steam API 키: 확인 필요 (상태 코드: {response.status_code})")
except Exception as e:
    print(f"❌ Steam API 키: 네트워크 오류 ({e})")

print("\n🚀 검사 종료!")

🔍 키 유효성 검사를 시작합니다...

✅ OpenAI API 키: 정상 (사용 가능)
------------------------------
⚠️ Steam API 키: 확인 필요 (상태 코드: 404)

🚀 검사 종료!
